# 03 Knowledge Mining and Clustering

This notebook covers:
1. Khai tac Quy tac Lien ket (Finding patterns in su tu chuc cua nhan vien)
2. Employee Clustering (Profiling different employee groups)

In [9]:
import sys
import os
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('../'))

from src.data.loader import load_config, load_data
from src.mining.association import run_association_mining
from src.mining.clustering import run_clustering, profile_clusters

In [10]:
import importlib
import src.mining.association
importlib.reload(src.mining.association)
from src.mining.association import run_association_mining

In [11]:
config = load_config('../configs/params.yaml')
df_disc = pd.read_csv('../data/processed/HR_Discretized.csv')
df_scaled = pd.read_parquet('../' + config['processed_data_path'])
df_clean = load_data('../' + config['raw_data_path']) # For profiling

## 1. Khai tac Quy tac Lien ket

In [12]:
resigned_rules, stayed_rules, frequent_itemsets = run_association_mining(df_disc)

print("Rules leading to Resignation (Top 10 by Lift):")
display(resigned_rules.sort_values('lift', ascending=False).head(10))

print("\nRules leading to Staying (Top 10 by Lift):")
display(stayed_rules.sort_values('lift', ascending=False).head(10))

Rules leading to Resignation (Top 10 by Lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1610,(overtime_Yes),"(department_Research & Development, left_Resig...",0.282417,0.090292,0.050238,0.177885,1.970106,1.0,0.024738,1.106545,0.686210,0.155789,0.096287,0.367138
1586,(overtime_Yes),"(businesstravel_Travel_Rarely, left_Resigned)",0.282417,0.105906,0.057026,0.201923,1.906620,1.0,0.027117,1.120310,0.662657,0.172131,0.107390,0.370192
210,(overtime_Yes),(left_Resigned),0.282417,0.160896,0.086219,0.305288,1.897426,1.0,0.040779,1.207845,0.659115,0.241445,0.172079,0.420577
1060,"(performancerating_3, overtime_Yes)",(left_Resigned),0.238289,0.160896,0.070604,0.296296,1.841538,1.0,0.032264,1.192411,0.599933,0.214876,0.161363,0.367557
1064,(overtime_Yes),"(performancerating_3, left_Resigned)",0.282417,0.135777,0.070604,0.250000,1.841250,1.0,0.032258,1.152297,0.636708,0.203125,0.132168,0.385000
1582,"(businesstravel_Travel_Rarely, overtime_Yes)",(left_Resigned),0.199593,0.160896,0.057026,0.285714,1.775769,1.0,0.024913,1.174745,0.545802,0.187919,0.148752,0.320072
1606,"(overtime_Yes, department_Research & Development)",(left_Resigned),0.183978,0.160896,0.050238,0.273063,1.697137,1.0,0.020636,1.154300,0.503384,0.170507,0.133674,0.292650
1207,(department_Sales),"(performancerating_3, left_Resigned)",0.303462,0.135777,0.055669,0.183445,1.351074,1.0,0.014465,1.058377,0.373057,0.145133,0.055157,0.296723
1204,"(department_Sales, performancerating_3)",(left_Resigned),0.262050,0.160896,0.055669,0.212435,1.320325,1.0,0.013506,1.065441,0.328763,0.151571,0.061422,0.279213
260,(department_Sales),(left_Resigned),0.303462,0.160896,0.062458,0.205817,1.279189,1.0,0.013632,1.056562,0.313342,0.155405,0.053534,0.297001



Rules leading to Staying (Top 10 by Lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
6292,"(department_Sales, overtime_No)","(performancerating_3, left_Stayed, educationfi...",0.216565,0.071283,0.058384,0.269592,3.781997,1.0,0.042947,1.271505,0.938928,0.254438,0.213530,0.544320
6296,"(educationfield_Marketing, overtime_No)","(department_Sales, performancerating_3, left_S...",0.075356,0.206382,0.058384,0.774775,3.754090,1.0,0.042832,3.523666,0.793413,0.261398,0.716205,0.528835
6283,"(department_Sales, performancerating_3, overti...","(educationfield_Marketing, left_Stayed)",0.190767,0.084182,0.058384,0.306050,3.635576,1.0,0.042325,1.319717,0.895837,0.269592,0.242262,0.499799
4309,"(educationfield_Marketing, overtime_No)","(department_Sales, left_Stayed)",0.075356,0.241005,0.065852,0.873874,3.625961,1.0,0.047691,6.017748,0.783233,0.262873,0.833825,0.573557
4307,"(department_Sales, overtime_No)","(educationfield_Marketing, left_Stayed)",0.216565,0.084182,0.065852,0.304075,3.612120,1.0,0.047621,1.315973,0.923056,0.280347,0.240106,0.543167
6286,"(performancerating_3, overtime_No, educationfi...","(department_Sales, left_Stayed)",0.067210,0.241005,0.058384,0.868687,3.604439,1.0,0.042186,5.780041,0.774627,0.233696,0.826991,0.555470
4433,"(department_Sales, businesstravel_Travel_Rarely)","(educationfield_Marketing, left_Stayed)",0.213849,0.084182,0.063815,0.298413,3.544854,1.0,0.045813,1.305352,0.913185,0.272464,0.233923,0.528239
6502,"(department_Sales, businesstravel_Travel_Rarely)","(performancerating_3, left_Stayed, educationfi...",0.213849,0.071283,0.052953,0.247619,3.473741,1.0,0.037709,1.234371,0.905839,0.228070,0.189871,0.495238
6493,"(department_Sales, performancerating_3, busine...","(educationfield_Marketing, left_Stayed)",0.182621,0.084182,0.052953,0.289963,3.444478,1.0,0.037580,1.289817,0.868238,0.247619,0.224696,0.459498
4439,(educationfield_Marketing),"(department_Sales, businesstravel_Travel_Rarel...",0.107943,0.173116,0.063815,0.591195,3.415020,1.0,0.045129,2.022685,0.792747,0.293750,0.505608,0.479911


## 2. Employee Clustering

In [13]:
df_clustered, kmeans, silhouette = run_clustering(df_scaled, n_clusters=3)
print(f"Silhouette Score: {silhouette:.4f}")

profile = profile_clusters(df_clean, df_clustered['cluster'])
print("\nCluster Profiles (Mean values):")
display(profile)

Silhouette Score: 0.0023

Cluster Profiles (Mean values):


,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,count
cluster,,,,,,,,,,,,,,,,,,,,,
0.0,47.621053,813.178947,9.273684,3.073684,1.0,1030.868421,2.742105,65.605263,2.736842,4.210526,...,80.0,0.800000,25.236842,2.678947,2.747368,15.142105,7.263158,5.189474,6.764706,190
1.0,33.076177,801.771468,9.130194,2.742382,1.0,1008.240997,2.717452,66.184211,2.746537,1.256233,...,80.0,0.749307,6.851801,2.832410,2.734072,4.518006,2.832410,1.277008,2.878261,722
2.0,38.235294,800.240642,9.256684,3.073084,1.0,1049.768271,2.725490,65.459893,2.705882,2.376114,...,80.0,0.848485,12.245989,2.800357,2.802139,7.449198,4.996435,2.333333,4.784787,561


## 3. Policy Suggestions based on Findings

Based on association rules and cluster profiles:
- If Cluster 0 has high satisfaction and low hours but also high attrition, perhaps pay or promotion is the issue.
- If Cluster 1 is 'workaholics' (high hours, high eval) and they are leaving, burnout is likely.